# Tele-Law Access Equity Analysis
### Problem Statement 37 – IBM Cloud Lite + IBM Granite AI

This notebook walks through the full analysis pipeline interactively.
Run all cells in order, or jump to any section independently.

In [ ]:
import sys, os
sys.path.insert(0, '..')  # add project root

from dotenv import load_dotenv
load_dotenv('../.env')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print('✅ Imports OK')

## 1. Load & Prepare Data

In [ ]:
# Option A: Generate demo data (no download needed)
from main import _generate_demo_data
from src.data_ingestion import clean_and_standardise

raw = _generate_demo_data()
df  = clean_and_standardise(raw)
print(f'Dataset shape: {df.shape}')
df.head()

In [ ]:
# Option B: Load from CSV
# from src.data_ingestion import load_and_prepare
# df = load_and_prepare('../data/raw/telelaw_data.csv')

# Option C: Fetch from data.gov.in API
# df = load_and_prepare()  # requires DATAGOV_RESOURCE_ID in .env

df.dtypes

## 2. Run Full Analysis

In [ ]:
from src.analysis import run_full_analysis
results = run_full_analysis(df)
print('Analysis modules:', list(results.keys()))

## 3. Gender Disparity

In [ ]:
gender = results['gender']
print('National gender distribution:')
for k, v in gender['national'].items():
    print(f'  {k}: {v}')

print(f"\nDistricts with female share < 30%: {len(gender['skewed_districts'])}")
gender['skewed_districts'].head(10)

In [ ]:
from src.visualisation import plot_gender_pie, plot_gender_trend
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('inline')

plot_gender_pie(gender['national'])
plt.show()

plot_gender_trend(gender['by_year'])
plt.show()

## 4. Caste / Marginalization Disparity

In [ ]:
caste = results['caste']
print('National caste distribution:')
for k, v in caste['national'].items():
    print(f'  {k}: {v}')
print(f"\nGini coefficient: {caste['gini_coefficient']}")
print(f"Underserved districts: {len(caste['underserved_districts'])}")
caste['by_state'][['sc_pct','st_pct','obc_pct','general_pct']].head(10)

## 5. Regional Disparity

In [ ]:
regional = results['regional']
print('State summary (top 10):')
display(regional['state_summary'].head(10))

print('\nTop 5 districts by cases/CSC:')
display(regional['district_top20'].head())

## 6. Composite Equity Score

In [ ]:
equity = results['equity']
print('Top 10 most equitable districts:')
display(equity[['district_name','state_name','equity_score','marginalized_pct','female_pct']].head(10))

print('\nBottom 10 least equitable districts:')
display(equity[['district_name','state_name','equity_score','marginalized_pct','female_pct']].tail(10))

## 7. IBM Granite AI Insights

In [ ]:
# ⚠️ Requires IBM_CLOUD_API_KEY and WATSONX_PROJECT_ID in .env
from src.granite_ai import run_granite_insights

insights = run_granite_insights(results)

for section, text in insights.items():
    print(f'\n{'='*60}')
    print(f'  {section.upper().replace("_", " ")}')
    print('='*60)
    print(text)

## 8. Generate Full Report

In [ ]:
import os, sys
sys.path.insert(0, '..')
os.chdir('..')  # run from project root

from src.visualisation import generate_all_charts, generate_html_report
from pathlib import Path

Path('reports').mkdir(exist_ok=True)
chart_paths = generate_all_charts(results, df)

# Use empty insights if IBM credentials not configured
empty_insights = {k: '[Configure IBM Granite in .env to generate AI insights]'
                  for k in ('gender_insight','caste_insight','regional_insight','executive_summary')}

report_path = generate_html_report(results, empty_insights, chart_paths)
print(f'\n✅ Report generated: {report_path.resolve()}')